In [1]:
import pandas as pd
df = pd.read_csv('vacancies_cleaned.csv')

In [4]:
df.head(2)

,title,link,tags,company,skills,salary,relocation,employment_type,onsite,remote,hybrid,lead,c_level,senior,director,trainee,head,middle,junior,location
0,Account Executive (SaaS),https://hirify.me/jobs/440679-account-executiv...,"['hybrid', 'US', 'fulltime', 'middle']",Company hidden,"['crm', 'sales', 'negotiation', 'saas', 'healt...",64 000 - 74 000\n$,False,fulltime,True,True,False,False,False,False,False,False,False,True,False,US
1,"Senior Account Manager, Key Accounts (SaaS)",https://hirify.me/jobs/440678-senior-account-m...,"['onsite', 'US', 'fulltime', 'senior']",Company hidden,"['sales', 'account management', 'salesforce', ...",NaN,False,fulltime,False,True,True,False,False,True,False,False,False,False,False,US


In [10]:
test = df[['title', 'location']]

In [11]:
test.head()

,title,location
0,Account Executive (SaaS),US
1,"Senior Account Manager, Key Accounts (SaaS)",US
2,Forward-Deployed Data Scientist II (AI),Australia
3,Customer Support Specialist (AI),Australia
4,Deal Strategy And Pricing Manager (Fintech),Italy


In [8]:
prompt_header1 = '''
You are a job title classification assistant.

Task:
Assign each vacancy title to exactly one category from the allowed list.

Allowed categories:
- qa
- development
- devops
- marketing
- analytics
- management
- design

Rules:
- Choose exactly one category per title.
- Do not invent new categories.
- Focus on the core job function.
- Ignore seniority words such as Junior, Middle, Senior, Lead, Head.
- Ignore company names, location, remote/hybrid markers, emojis, and salary mentions.
- If a title is ambiguous, choose the closest category based on the main function.
- If the title is too vague to classify reliably, use Other.

Responce format:
[{"id": 1, "category": "development"}, {"id": 2, "category": "marketing"}, ...]

## Job Titles to Classify:
'''
def cook_promt(objects, prompt_header):
    prompt = prompt_header
    for i, obj in enumerate(objects):
        prompt += f"{i+1}. {obj}\n"
    return prompt

In [15]:
import json
import time
import requests

def cook_promt(objects, prompt_header):
    prompt = prompt_header
    for i in objects:
        prompt += f"{i['id']}. {i['title']}\n"
    return prompt


def send_request(prompt):
    headers = {
        'Content-Type': 'application/json',
        'Authorization': 'Bearer ***'
    }
    system_prompt ='You must answer in json format to user question.'
    data = {
        'model': 'qwen/qwen3-vl-8b',
        'input': prompt,
        'system_prompt': system_prompt,
    }
    r = requests.post('http://127.0.0.1:1234/api/v1/chat', headers=headers, json=data)
    return r


def predict(df, batch_size=10):
    new_df = df.copy()
    titles = [{'id': i, 'title': df['title'].iloc[i]} for i in range(len(df))]
    batches = [titles[i:i+batch_size] for i in range(0, len(titles), batch_size)]
    new_df['predict_category'] = None
    for batch in batches:
        print(f'Batch: {batch[0]['id']}')
        try:
            prompt = cook_promt(batch, prompt_header1)
            answer = send_request(prompt)
            answer = answer.json()
            answer = answer['output']
            answer = answer[0]['content']
            answer = json.loads(answer)
            indexes = [i['id'] for i in batch]
            for i in range(len(indexes)):
                if indexes[i] == answer[i]['id']:
                    new_df.loc[indexes[i], 'predict_category'] = answer[i]['category']
        except Exception as e:
            print(f'Error on batch{batch}')
    return new_df



test = test.reset_index(drop=True)
d = predict(test, 150)

Batch: 0
Batch: 150
Batch: 300
Batch: 450
Batch: 600
Batch: 750
Batch: 900
Batch: 1050
Batch: 1200
Batch: 1350
Batch: 1500
Batch: 1650
Batch: 1800
Batch: 1950
Batch: 2100
Batch: 2250
Batch: 2400
Batch: 2550
Batch: 2700
Batch: 2850
Batch: 3000
Batch: 3150
Batch: 3300
Batch: 3450
Batch: 3600
Batch: 3750
Batch: 3900
Batch: 4050
Batch: 4200
Batch: 4350
Batch: 4500
Batch: 4650
Batch: 4800
Batch: 4950
Batch: 5100
Batch: 5250
Batch: 5400
Batch: 5550
Batch: 5700
Batch: 5850
Batch: 6000
Batch: 6150
Batch: 6300
Batch: 6450
Batch: 6600
Batch: 6750
Batch: 6900
Batch: 7050
Batch: 7200
Batch: 7350
Batch: 7500
Batch: 7650
Batch: 7800
Batch: 7950
Batch: 8100
Batch: 8250
Batch: 8400
Batch: 8550
Batch: 8700
Batch: 8850
Batch: 9000
Batch: 9150
Batch: 9300
Batch: 9450
Batch: 9600
Batch: 9750
Batch: 9900
Error on batch[{'id': 9900, 'title': 'Директор по МСФО'}, {'id': 9901, 'title': 'VIP Account Manager (Marketing)'}, {'id': 9902, 'title': 'Ассистент по комментингу (Маркетинг)'}, {'id': 9903, 'title': 'Lea

In [16]:
d.head(10)
d.to_csv('predicted.csv')

In [18]:
d['predict_category'].value_counts()

predict_category
management          3381
development         2972
marketing           1548
analytics            834
design               693
devops               395
qa                   345
Other                151
other                104
sales                 69
customer support      28
security              27
customer service       3
education              2
Name: count, dtype: int64